# Prepare Demo 

Notebook to run to prepare data ready for demo.

In [16]:
import polars as pl
from pathlib import Path
import xlsxwriter
import duckdb
import json
import uuid

from data_importers import WorldBankDataImporter
importer = WorldBankDataImporter()

In [17]:
DATA_EXPORT_PATH = Path("../data/world_bank")
START_YEAR = 1970
END_YEAR = 2024

SELECTED_COUNTRIES = [
    'GBR', 'FRA', 'DEU', 'ALB', 'NOR', 'GRC',   #Europe & Central Asia
    'JPN', 'CHN',                               #East Asia & Pacific
    'IND', 'PAK', 'BGD',                        #South Asia
    'SAU', 'EGY', 'ISR',                        #Middle East & North Africa
    'NGA', 'ZAF', 'TZA',                        #Sub-Saharan Africa
    'USA', 'CAN', 'MEX',                        #North America
    'CUB', 'DOM', 'CRI', 'BRA', 'ARG', 'CHL',   #Latin America & Caribbean
    'AUS', 'NZL', 'PHL', 'IDN', 'VNM', 'THA'    #East Asia & Pacific
]

SELECTED_INDICATORS = [
    'SP.POP.TOTL',          # Population, total
    'NY.GDP.MKTP.CD',       # GDP (current US$)
    'SE.ADT.LITR.ZS',       # Literacy rate, adult total
    'SL.UEM.TOTL.ZS',       # Unemployment, total (% of  total labor force)
    'SP.DYN.LE00.IN',       # Life expectancy at birth, total (years)
    'EG.USE.PCAP.KG.OE',    # Energy use per capita (kg of oil equivalent)
    'GC.DOD.TOTL.GD.ZS',    # Central government debt, total (% of GDP)
    'NY.GDP.PCAP.CD',       # GDP (US$, per capita)
]

## Get List of Countries

In [18]:
countries = importer.get_countries()

INFO:root:Fetching countries page 1...


In [19]:
countries.columns

['country_code',
 'iso2_code',
 'country_name',
 'region',
 'region_id',
 'income_level',
 'capital_city',
 'longitude',
 'latitude']

In [20]:
countries.filter(pl.col("country_code").is_in(SELECTED_COUNTRIES))

country_code,iso2_code,country_name,region,region_id,income_level,capital_city,longitude,latitude
str,str,str,str,str,str,str,str,str
"""ALB""","""AL""","""Albania""","""Europe & Central Asia""","""ECS""","""Upper middle income""","""Tirane""","""19.8172""","""41.3317"""
"""ARG""","""AR""","""Argentina""","""Latin America & Caribbean ""","""LCN""","""Upper middle income""","""Buenos Aires""","""-58.4173""","""-34.6118"""
"""AUS""","""AU""","""Australia""","""East Asia & Pacific""","""EAS""","""High income""","""Canberra""","""149.129""","""-35.282"""
"""BGD""","""BD""","""Bangladesh""","""South Asia""","""SAS""","""Lower middle income""","""Dhaka""","""90.4113""","""23.7055"""
"""BRA""","""BR""","""Brazil""","""Latin America & Caribbean ""","""LCN""","""Upper middle income""","""Brasilia""","""-47.9292""","""-15.7801"""
…,…,…,…,…,…,…,…,…
"""THA""","""TH""","""Thailand""","""East Asia & Pacific""","""EAS""","""Upper middle income""","""Bangkok""","""100.521""","""13.7308"""
"""TZA""","""TZ""","""Tanzania""","""Sub-Saharan Africa ""","""SSF""","""Lower middle income""","""Dodoma""","""35.7382""","""-6.17486"""
"""USA""","""US""","""United States""","""North America""","""NAC""","""High income""","""Washington D.C.""","""-77.032""","""38.8895"""


## Download indicator data

For each country, for each indicator between the years chosen.

In [21]:
data = importer.get_data(
    indicators=SELECTED_INDICATORS,
    countries=SELECTED_COUNTRIES,
    start_year=START_YEAR,
    end_year=END_YEAR
)

INFO:root:Fetching data with skip=0...
INFO:root:Fetching data with skip=1000...
INFO:root:Fetching data with skip=2000...
INFO:root:Fetching data with skip=3000...
INFO:root:Fetching data with skip=4000...
INFO:root:Fetching data with skip=5000...
INFO:root:Fetching data with skip=6000...
INFO:root:Fetching data with skip=7000...
INFO:root:Fetching data with skip=8000...
INFO:root:Fetching data with skip=9000...


In [22]:
data

country_code,year,indicator_code,value
str,i64,str,f64
"""VNM""",2011,"""WB_WDI_EG_USE_PCAP_KG_OE""",663.57444
"""VNM""",2018,"""WB_WDI_EG_USE_PCAP_KG_OE""",885.646842
"""GBR""",2016,"""WB_WDI_EG_USE_PCAP_KG_OE""",2696.97999
"""CAN""",2010,"""WB_WDI_EG_USE_PCAP_KG_OE""",7660.754521
"""CHL""",2010,"""WB_WDI_EG_USE_PCAP_KG_OE""",1795.68173
…,…,…,…
"""BGD""",1970,"""WB_WDI_SP_POP_TOTL""",6.9058894e7
"""BRA""",1970,"""WB_WDI_SP_POP_TOTL""",9.5375651e7
"""CAN""",1970,"""WB_WDI_SP_POP_TOTL""",2.1324e7


## Capture indicator metadata

In [23]:
indicators = importer.get_indicator_metadata(indicators=SELECTED_INDICATORS)
indicators

INFO:root:Fetching metadata for 8 indicators...


indicator_code,indicator_name,aggregation_method,definition_long,statistical_concept,methodology,limitation,relevance
str,str,str,str,str,str,str,str
"""WB_WDI_GC_DOD_TOTL_GD_ZS""","""Central government debt, total…","""Weighted average""","""Debt is the entire stock of di…","""Government Financial Statistic…","""Government Finance statistics …","""For most countries central gov…","""This indicator is related to G…"
"""WB_WDI_NY_GDP_PCAP_CD""","""GDP per capita (current US$)""","""Weighted average""","""Gross domestic product is the …","""The conceptual elements of the…","""National accounts are compiled…",null,"""This indicator is related to t…"
"""WB_WDI_SP_POP_TOTL""","""Population, total""","""Sum""","""Total population is based on t…","""Estimates of total population …","""Population estimates are usual…","""Current population estimates f…","""Increases in human population,…"
"""WB_WDI_SL_UEM_TOTL_ZS""","""Unemployment, total (% of tota…","""Weighted average""","""Unemployment refers to the sha…","""The unemployed comprise all pe…","""The unemployment rate is calcu…","""While the unemployment rate ma…","""The unemployment rate is a use…"
"""WB_WDI_SP_DYN_LE00_IN""","""Life expectancy at birth, tota…","""Weighted average""","""Life expectancy at birth indic…","""Life expectancy at birth used …","""Life expectancy at birth is de…","""Annual data series from United…","""Mortality rates for different …"
"""WB_WDI_SE_ADT_LITR_ZS""","""Literacy rate, adult total (% …","""Weighted average""","""Adult literacy rate is the per…","""Literacy statistics for most c…","""The indicator is calculated by…","""In practice, literacy is diffi…","""Literacy rate is an outcome in…"
"""WB_WDI_EG_USE_PCAP_KG_OE""","""Energy use (kg of oil equivale…","""Weighted average""","""Energy use refers to use of pr…",null,"""Total energy use refers to the…","""The IEA makes these estimates …","""In developing economies growth…"
"""WB_WDI_NY_GDP_MKTP_CD""","""GDP (current US$)""","""Gap-filled total""","""Gross domestic product is the …","""The conceptual elements of the…","""National accounts are compiled…","""Gross domestic product (GDP), …","""This indicator is related to t…"


## Export data in a range of formats

### CSV

In [24]:
CSV_PATH = DATA_EXPORT_PATH / "csv"
CSV_PATH.mkdir(parents=True, exist_ok=True)

indicators.write_csv(CSV_PATH / "indicators.csv")
data.write_csv(CSV_PATH / "data.csv")
countries.write_csv(CSV_PATH / "countries.csv")

### Parquet

In [25]:
PARQUET_PATH = DATA_EXPORT_PATH / "parquet"
PARQUET_PATH.mkdir(parents=True, exist_ok=True)

# Cleat any files below the parquet path
for f in PARQUET_PATH.glob("**/*.parquet"):
    f.unlink()

# Let's pivot the data file so each indicator is a column
data_pivoted = data.pivot(
    values="value",
    index=["country_code", "year"],
    on="indicator_code"
)

# Now join with countries to get country names and regions
data_pivoted = data_pivoted.join(
    countries.select(["country_code", "country_name", "region"]),
    on="country_code",
    how="left"
)

# Write out individual parquet files, one for each year and name them accordingly
for year in range(START_YEAR, END_YEAR + 1):
    unique_file_id = uuid.uuid4().hex
    folder = PARQUET_PATH / f"year={year}"
    folder.mkdir(parents=True, exist_ok=True)
    year_data = data_pivoted.filter(pl.col("year") == year)
    year_data.write_parquet(folder / f"{unique_file_id}.parquet")

## Excel

In [26]:
# Write multiple worksheets to a single Excel file using Polars
EXCEL_PATH = DATA_EXPORT_PATH / "excel"
EXCEL_PATH.mkdir(parents=True, exist_ok=True)
with xlsxwriter.Workbook(EXCEL_PATH / "world_bank_data.xlsx") as workbook:
    data.write_excel(workbook, worksheet="Raw Data")
    countries.write_excel(workbook, worksheet="Countries")
    indicators.write_excel(workbook, worksheet="Indicators")
    

## JSON

In [27]:
JSON_PATH = DATA_EXPORT_PATH / "json"
JSON_PATH.mkdir(parents=True, exist_ok=True)

# Just focus on last 5 years
data_for_json = data.filter(pl.col("year") <= (END_YEAR - 5))

# Add complex column for year: value
data_for_json = data_for_json.drop_nulls("value").with_columns(pl.struct(["year", "value"]).alias("data_points"))

# Unexplode data by country, year and indicator_code
data_for_json = data_for_json.group_by(["country_code", "indicator_code"]).agg(pl.col("data_points").implode())

# Join to country
data_for_json = (
    data_for_json
    .join(
        countries.select(["country_code", "country_name", "region", "income_level", "capital_city", "latitude", "longitude"]),
        on="country_code",
        how="left"
    )
    .with_columns(pl.struct(["country_name", "region", "income_level", "capital_city", "latitude", "longitude"]).alias("country_info"))
)

# Join to indicator
data_for_json = (
    data_for_json
    .join(
        indicators.select(["indicator_code", "indicator_name", "aggregation_method"]),
        on="indicator_code",
        how="left"
    )
    .with_columns(pl.struct(["indicator_code", "indicator_name", "aggregation_method", "data_points"]).alias("indicators"))
)

data_for_json = data_for_json.group_by(["country_code", "country_info"]).agg(pl.col("indicators").implode())

list_of_countries = data_for_json["country_code"].to_list()

for country_code in list_of_countries:
    data_for_json.filter(pl.col("country_code") == country_code).write_ndjson(JSON_PATH / f"{country_code}_data.json")


## DuckDB

In [28]:
DUCKDB_PATH = DATA_EXPORT_PATH / "duckdb" / "world_bank.db"
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

if DUCKDB_PATH.exists():
    # Remove the file, we want to start from scratch
    DUCKDB_PATH.unlink()

with duckdb.connect(DUCKDB_PATH) as db:

    db.sql("CREATE SCHEMA IF NOT EXISTS health_wealth_analytics")

    db.sql(
        f"""
        CREATE TABLE IF NOT EXISTS countries AS
        SELECT *
        FROM countries;
        """
    )

    db.sql(
        f"""
        CREATE TABLE IF NOT EXISTS data AS
        SELECT *
        FROM data;
        """
    )

    db.sql(
        f"""
        CREATE TABLE IF NOT EXISTS indicators AS
        SELECT *
        FROM indicators;
        """
    )

db.close()